# ML - BOOSTING

In [ ]:
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from utils1 import get_classifier_metrics
from sklearn.model_selection import GridSearchCV
from collections import Counter
from sklearn.metrics import classification_report

## Paso 1. Lectura del conjunto de datos procesado

In [2]:
# Cargamos los dataframes, el dataframe de los datos invalidatos imputados con la moda no lo cargamos ya que en modelos anteriores es el que peor metricas tiene y para predicciones de salud no es recomendable.
with open('../data/processed/04_df_invalids_removed.pkl', 'rb') as f:
    df_invalids_removed = pickle.load(f)

with open('../data/processed/04_df_invalids_knn.pkl', 'rb') as f:
    df_invalids_knn = pickle.load(f)

## Paso 2. Split

In [3]:
X_invalids_removed = df_invalids_removed.drop('Outcome', axis= 1)
y_invalids_removed = df_invalids_removed['Outcome']

X_invalids_knn= df_invalids_knn.drop('Outcome', axis= 1)
y_invalids_knn = df_invalids_knn['Outcome']

X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X_invalids_removed, y_invalids_removed, test_size=0.2, random_state=21)
X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(X_invalids_knn, y_invalids_knn, test_size=0.2, random_state=21)

## Paso 3. Modelado y Ajuste

In [4]:
default_bm_1 = XGBClassifier(random_state=21)
default_bm_1.fit(X_train_1, y_train_1)

default_bm_3 = XGBClassifier(random_state=21)
default_bm_3.fit(X_train_3, y_train_3)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


## Paso 4. Predicción

In [5]:
y_pred_test_1 = default_bm_1.predict(X_test_1)
y_pred_train_1 = default_bm_1.predict(X_train_1)

y_pred_test_3 = default_bm_3.predict(X_test_3)
y_pred_train_3 = default_bm_1.predict(X_train_3)

In [6]:
default_bm_metrics_1 = get_classifier_metrics(y_pred_test_1, y_test_1, y_pred_train_1, y_train_1, average='weighted')
default_bm_metrics_1

,Accuracy,F1 Score,Precision,Recall
Train set,1.000000,1.000000,1.000000,1.000000
Test set,0.734177,0.723725,0.726029,0.734177


In [7]:
report_bm_1 = classification_report(y_test_1, y_pred_test_1)
print(report_bm_1)

              precision    recall  f1-score   support

           0       0.76      0.86      0.81        51
           1       0.67      0.50      0.57        28

    accuracy                           0.73        79
   macro avg       0.71      0.68      0.69        79
weighted avg       0.73      0.73      0.72        79



In [8]:
default_bm_metrics_3 = get_classifier_metrics(y_pred_test_3, y_test_3, y_pred_train_3, y_train_3, average='weighted')
default_bm_metrics_3

,Accuracy,F1 Score,Precision,Recall
Train set,0.827362,0.820225,0.827312,0.827362
Test set,0.701299,0.685789,0.696685,0.701299


In [9]:
report_bm_3 = classification_report(y_test_3, y_pred_test_3)
print(report_bm_3)

              precision    recall  f1-score   support

         0.0       0.71      0.86      0.78        94
         1.0       0.68      0.45      0.54        60

    accuracy                           0.70       154
   macro avg       0.69      0.66      0.66       154
weighted avg       0.70      0.70      0.69       154



### Modelo optimizado con los hiperparámetros con `GridSearchCV`

'''n_estimators'# número de árboles
    'max_depth' # profundidad máxima de cada árbol
    'learning_rate': # tasa de aprendizaje
    'subsample': # proporción de muestras usadas para entrenar cada árbol
    'colsample_bytree':  # proporción de características usadas por árbol
    'gamma': # regularización: requiere mejora mínima de pérdida para dividir
    'reg_alpha': # L1 regularization (sparse features)
    'reg_lambda': # L2 regularization (default is 1)

In [68]:
#Calcular manualmente el peso para balancear clases (0: no diabetes, 1: diabetes)
counts_1 = Counter(y_train_1)
scale_pos_weight_1 = counts_1[0] / counts_1[1]

counts_3 = Counter(y_train_3)
scale_pos_weight_3 = counts_3[0] / counts_3[1]

scale_pos_weight_1

2.0784313725490198

In [ ]:
param_grid_3 = {'n_estimators': list(range(15, 31, 5)),
              'max_depth' : [2, 3, 5],
              'learning_rate': [0.005, 0.01, 0.015],
              'subsample': [0.8, 1.0],
              'scale_pos_weight': [1, scale_pos_weight_3],
              'colsample_bytree': [0.8, 1.0]}

param_grid_1 = {'n_estimators': list(range(15, 31, 5)),
                'max_depth' : [2, 3, 5],
                'learning_rate': [0.005, 0.01, 0.015],
                'subsample': [0.8, 1.0],
                'scale_pos_weight': [scale_pos_weight_1, scale_pos_weight_1 + 0.04],
                'colsample_bytree': [0.8, 1.0]}

grid_1 = GridSearchCV(default_bm_1,
                      param_grid_1,
                      scoring='recall',
                      cv=5)

grid_3 = GridSearchCV(default_bm_3,
                      param_grid_3,
                      scoring='recall',
                      cv=5)    

In [209]:
# Entrenamos el grid con los hiperparametros
grid_1.fit(X_train_1, y_train_1)
#Devuelvemos los mejores parametros despues de entrenarlo
grid_1.best_params_

{'colsample_bytree': 0.8,
 'learning_rate': 0.005,
 'max_depth': 3,
 'n_estimators': 20,
 'scale_pos_weight': 2.11343137254902,
 'subsample': 1.0}

In [25]:
# Entrenamos el grid con los hiperparametros
grid_3.fit(X_train_3, y_train_3)
#Devuelvemos los mejores parametros despues de entrenarlo
grid_3.best_params_

{'colsample_bytree': 0.8,
 'gamma': 5,
 'learning_rate': 0.1,
 'max_depth': 3,
 'n_estimators': 25,
 'scale_pos_weight': 1.9519230769230769,
 'subsample': 1.0}

In [210]:
#Modelos con los mejores parametros
grid_bm_1 = grid_1.best_estimator_
#grid_bm_3 = grid_3.best_estimator_

In [211]:
# Repetimos el entrenamiento pero ahora con el grid que tiene los hiperparametros establecidos
grid_bm_1.fit(X_train_1, y_train_1)
#grid_bm_3.fit(X_train_3, y_train_3)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [212]:
y_pred_test_1_grid = grid_bm_1.predict(X_test_1)
y_pred_train_1_grid = grid_bm_1.predict(X_train_1)

#y_pred_test_3_grid = grid_bm_3.predict(X_test_3)
#y_pred_train_3_grid = grid_bm_3.predict(X_train_3)

In [213]:
grid_boosting_model_metrics_1 = get_classifier_metrics(y_pred_test_1_grid, y_test_1, y_pred_train_1_grid, y_train_1, average='weighted')
grid_boosting_model_metrics_1 


,Accuracy,F1 Score,Precision,Recall
Train set,0.821656,0.826621,0.850793,0.821656
Test set,0.835443,0.837133,0.841287,0.835443


In [215]:
report_bm_1 = classification_report(y_test_1, y_pred_test_1_grid)
print(report_bm_1)

              precision    recall  f1-score   support

           0       0.90      0.84      0.87        51
           1       0.74      0.82      0.78        28

    accuracy                           0.84        79
   macro avg       0.82      0.83      0.82        79
weighted avg       0.84      0.84      0.84        79



In [32]:
grid_boosting_model_metrics_3 = get_classifier_metrics(y_pred_test_3_grid, y_test_3, y_pred_train_3_grid, y_train_3, average='weighted')
grid_boosting_model_metrics_3

NameError: name 'y_pred_test_3_grid' is not defined

In [128]:
report_bm_3 = classification_report(y_test_3, y_pred_test_3_grid)
print(report_bm_3)

              precision    recall  f1-score   support

         0.0       0.79      0.82      0.81        94
         1.0       0.70      0.67      0.68        60

    accuracy                           0.76       154
   macro avg       0.75      0.74      0.75       154
weighted avg       0.76      0.76      0.76       154



In [ ]:

param_grid_1 = {'n_estimators': list(range(15, 31, 5)),
              'max_depth' : [2, 3, 5],
              'learning_rate': [0.005, 0.01, 0.015],
              'subsample': [0.8, 1.0],
              'scale_pos_weight': [1, scale_pos_weight_1],
              'colsample_bytree': [0.8, 1.0]}

param_grid_1 = {'n_estimators': list(range(15, 31, 5)),
              'max_depth' : [2, 3, 5],
              'learning_rate': [0.005, 0.01, 0.015],
              'subsample': [0.8, 1.0],
              'scale_pos_weight': [scale_pos_weight_1, scale_pos_weight_1 + 0.04],
              'colsample_bytree': [0.8, 1.0]}